### Draw main graph

In [6]:
import numpy as np
import tensorflow.compat.v1 as tf
from config import *
from GPT_Model import *
from data_pipeline import get_data, data_train_files, data_test_files

tf.reset_default_graph()

tf.compat.v1.disable_eager_execution()
X = tf.placeholder(tf.int32, [None, hparams.n_time])
Y = tf.placeholder(tf.int32, [None, hparams.n_time])

X_onehot = tf.one_hot(X, axis=2, depth=hparams.n_vocab[0])

logits = model(hparams, X)['logits']
probs = tf.nn.softmax(logits, axis=2)
cross_entropy = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=Y, logits=logits)
loss = tf.reduce_mean(cross_entropy)

temperature = 0
u = tf.random.uniform(shape=tf.shape(logits[:, -1]), minval=1e-5, maxval=1.-1e-5)
u = (logits[:, -1] - tf.log(temperature + 1e-8)) - tf.log(-tf.log(u))
sample = tf.argmax(u, axis=1)

'''
Train
'''
global_step = tf.Variable(0, name='global_step')
learning_rate = tf.train.exponential_decay(1e-3, global_step, 100, 0.7, staircase=False)

if mode == "pretrain":
    train_step = tf.train.AdamOptimizer(learning_rate).minimize(loss, global_step)
elif mode == "finetune":
    optimizer = tf.train.AdamOptimizer(learning_rate)
    output_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope='linear1|linear2')
    train_step = optimizer.minimize(loss, var_list=output_vars, global_step=global_step)

'''
Session Open
'''

# GPU number to use
gpu_options = tf.GPUOptions(visible_device_list="0")
sess = tf.Session(config=tf.ConfigProto(gpu_options=gpu_options))

sess.run(tf.global_variables_initializer())

print('graph create')

graph create


### Load model if exist && TensorboardX Logger

In [8]:
import tf_slim as slim
from tensorflow.python import pywrap_tensorflow

load_dir = '../music_save_model' 
save_dir = '../music_save_model'

#只恢复transformer部分的参数
sess.run(tf.global_variables_initializer())
ref_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope='transformer')
saver = tf.train.Saver(ref_vars)

restore_file = tf.train.latest_checkpoint(load_dir)
print(restore_file)
if restore_file is not None:
    saver.restore(sess, restore_file)
    print("Model restored.", restore_file)
else:
    print('model not exist.')

#Logger
from tensorboardX import SummaryWriter

class Logger(SummaryWriter):
    def __init__(self, logdir):
        super(Logger, self).__init__(logdir)

    def log(self, log_string, value, iteration):
            self.add_scalar(log_string, value, iteration)
            
logger = Logger(save_dir)  
        

../music_save_model/checkpoint-800
INFO:tensorflow:Restoring parameters from ../music_save_model/checkpoint-800


NotFoundError: Restoring from checkpoint failed. This is most likely due to a Variable name or other graph key that is missing from the checkpoint. Please ensure that you have not altered the graph expected based on the checkpoint. Original error:

2 root error(s) found.
  (0) Not found: Key beta1_power not found in checkpoint
	 [[node save/RestoreV2 (defined at <ipython-input-8-434e2803e3f9>:13) ]]
	 [[save/RestoreV2/_135]]
  (1) Not found: Key beta1_power not found in checkpoint
	 [[node save/RestoreV2 (defined at <ipython-input-8-434e2803e3f9>:13) ]]
0 successful operations.
0 derived errors ignored.

Original stack trace for 'save/RestoreV2':
  File "/home/sdmda/anaconda3/envs/hrsoup/lib/python3.7/runpy.py", line 193, in _run_module_as_main
    "__main__", mod_spec)
  File "/home/sdmda/anaconda3/envs/hrsoup/lib/python3.7/runpy.py", line 85, in _run_code
    exec(code, run_globals)
  File "/home/sdmda/anaconda3/envs/hrsoup/lib/python3.7/site-packages/ipykernel_launcher.py", line 16,

### Train

In [3]:
from IPython.display import clear_output
from tqdm import tqdm_notebook as tqdm
import matplotlib.pyplot as plt
from time import sleep
import time
import math

print('iteration\t', 'loss\t', 'train_perplexity\t')
while(True):
    for _ in range(100):
        _inputs = []
        _targets = []
        for _ in range(batch_size):
            while(True):
                x, y = get_data(hparams.n_time, data_train_files,'train', 0, type, EventDim)
                if(x.shape == y.shape):
                    break
                 
            _inputs.append(x)
            _targets.append(y)
        _inputs = np.stack(_inputs)
        _targets = np.stack(_targets)
#         print(_inputs.shape, _targets.shape)
        
        _, _global_step, _loss = sess.run([train_step, global_step, loss], 
                                          feed_dict={X: _inputs, 
                                                     Y: _targets})
        
        train_perplexity = math.exp(_loss) #log perplexity和交叉熵等价
        
        if _global_step % 10 == 0:
            logger.log('loss', _loss, _global_step)
            print(str(_global_step)+'\t', str(_loss)+'\t', str(train_perplexity)+'\t')
        
        if _global_step % 100 == 0:
            save_path = saver.save(sess, save_dir + '/checkpoint', global_step=_global_step)
            print("Model saved in path: %s" % save_path)

iteration	 loss	 train_perplexity	
0	 5.8865843	 360.1729316915826	
Model saved in path: ../music_save_model/checkpoint-0
10	 4.905934	 135.08900492836366	
20	 4.5886707	 98.36359150813301	
30	 4.417848	 82.91766356551278	
40	 4.3938613	 80.95239726200775	
50	 4.3651633	 78.66224650345214	
60	 4.413168	 82.53050259987297	
70	 4.4233027	 83.37117702839788	
80	 4.3861027	 80.3267488059129	
90	 4.3413863	 76.81395414202504	


### Compute perplexity on test set

In [5]:
import math

inputs = []
targets = []

l = len(data_test_files)

for i in range(l): 
    while(True):
        x_test, y_test = get_data(hparams.n_time, data_test_files, 'test', i, 'music', EventDim)
        if(x_test.shape == y_test.shape):
            break       
    inputs.append(x_test)
    targets.append(y_test)
    
inputs = np.stack(inputs)
targets = np.stack(targets)

test_loss = sess.run(loss, feed_dict={X: inputs, Y: targets})
test_perplexity = math.exp(test_loss)
test_perplexity

360.8667572431745